Define gene subset for GRN Inference

In [1]:
from anndata import AnnData 
import cellrank as cr
import math
from matplotlib import pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
from numpy.linalg import pinv
import ot 
import pandas as pd
import random
import seaborn as sbn
import scipy
import anndata as ad
import scanpy as sc
import scFates as scf
import scipy.stats as stats
from scipy import sparse
from scipy.stats import wasserstein_distance
from scipy.special import gammaln
from scipy.cluster.hierarchy import linkage, to_tree, dendrogram
from scipy.spatial.distance import pdist
import scvelo as scv
from joblib import Parallel, delayed

/Users/olivier_2/carda_venv/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
# Hyperparameter values
SFT= 12.5 # time scale factor
s=30 # point size for scatter plots
f=10 # stabilization factor for mRNA degradation rates
DRR=10 # degradation rate ratio limit
CC = 20  # cell cycle time

path="/Users/olivier_2/Documents/En_cours/Labo/Manipes/OG3700/"

1. Load the Anndata object

In [3]:
adata = ad.read_h5ad(path+'adata_3700_2.h5ad')

2. Generates the file for computing entropy

In [4]:
df_gene_expression = pd.DataFrame(
    adata.layers['counts'].toarray(), 
    index=adata.obs.index,
    columns=adata.var.index
)

df_gene_expression['time'] = adata.obs['time'].values
times = df_gene_expression['time'].values
gene_columns = [c for c in df_gene_expression.columns if c != 'time']

counts_matrix = df_gene_expression[gene_columns].values.astype(np.float32)

# Save the matrix to a CSV file
df_gene_expression.transpose().to_csv("gene_expression_by_time.csv")    

unique_times = np.unique(times)
print(f"Detected time points: {unique_times}")
print(f"Matrix shape: {counts_matrix.shape[0]} cells x {counts_matrix.shape[1]} genes")

Detected time points: [ 0.  12.5 25.  37.5 50. ]
Matrix shape: 4345 cells x 30034 genes


3. Compute entropy using bub_H.R